# Simple RAG - Query and Retrieval

**Flow:**

Query → Retrieve → LLM → Answer

## Install Dependencies

In [1]:
# # Conda environment setup
# !pip install langchain langchain-chroma langchain-openai chromadb

## Basic RAG Code Phase 2 - Query and Retrieval

In [2]:
# # Colab setup
# from google.colab import drive
# from google.colab import userdata
# drive.mount('/content/drive')

In [3]:
# # Colab Store Location
# store_location = "/content/drive/MyDrive/rag_langchain/data/chroma_db1"

In [4]:
# # Colab Key
# from google.colab import userdata
# import os
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [6]:
# Local setup
from pathlib import Path

store_location = "../vector_db/chroma_db2"

if not Path(store_location).exists():
    raise FileNotFoundError(f"Vector store not found: {store_location}")

In [7]:
# Use Python dotenv to load environment variables from a .env file
from dotenv import load_dotenv
import os

load_dotenv()

True

### Step 1: Create retriever

In [8]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embedding = OpenAIEmbeddings()

vectorstore = Chroma(
    persist_directory=store_location,
    embedding_function=embedding
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

### Step 2: Build Modern RAG Chain (LCEL)

In [9]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.
Answer the question based only on the context below.

Context:
{context}

Question:
{question}
""")

### Step 3: Compose the chain

In [10]:
def format_docs(docs):
    formatted = []
    for doc in docs:
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page")
        location = f"{source} (page {page})" if page else source
        formatted.append(f"Source: {location}\n{doc.page_content}")
    return "\n\n".join(formatted)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

### Step 4: Query

In [11]:
question = "What are the main topics across these documents?"
response = rag_chain.invoke(question)
print(response)

The main topics across these documents include:

1. **SpinRite and its Use in Space**: Discussion about SpinRite being used on the International Space Station (ISS) to address issues caused by cosmic rays affecting magnetic domains in storage.

2. **Bit Flips in RAM**: The phenomenon of bit flips in non-ECC (Error-Correcting Code) RAM, which can lead to crashes and failures in software like Firefox. This is attributed to cosmic rays striking the RAM.

3. **Malware Warnings on Macs**: A cautionary message regarding potential malware when users attempt to paste text into the Terminal on Macs, highlighting the risks of scams and privacy compromise.

Overall, the documents touch on computer reliability, data integrity issues, and security warnings related to software and hardware.


## Debug

In [12]:
docs = retriever.invoke(question)

for i, d in enumerate(docs):
    source = d.metadata.get("source", "unknown")
    page = d.metadata.get("page")
    location = f"{source} (page {page})" if page else source
    print(f'chunk {i} | {location}: {d.page_content}')
    print("-" * 50)

chunk 0 | ../data/sn-1069.pdf (page 3): Steve: Well, SpinRite was up in space. There is a copy on the Space Station. 
Leo: No.
Steve: The ISS has it. Yeah.
Leo: I didn't know that.
Steve: Oh, yeah. They use it all the time, apparently. You know, those pesky neutrons, 
they'll mess up your magnetic domains. And so SpinRite puts them back where they're 
supposed to be.
Leo: We had a story this week that somebody did some instrumentation at Firefox. 
10% of Firefox crashes and failures came from bit flips. You know, in non-ECC RAM,
--------------------------------------------------
chunk 1 | ../data/sn-1072.pdf (page 25): an intercept dialog will be displayed to caution the user about the possible implications of 
what they are attempting to do. The dialog reads: "Possible malware, Paste blocked." 
And it says: "Your Mac has not been harmed. Scammers often encourage pasting text 
into Terminal to try and harm your Mac or compromise your privacy. These instructions 
are commonly offered vi